# Multi-Criteria Ranking and Robustness

Compares equal weighting, entropy-TOPSIS, Borda aggregation and domain-prior weighting, with rank correlation and robustness checks across reconstruction cohorts.

> Public portfolio version; research data are not included.


In [ ]:
# Cell 0 — Imports, constants, helpers
import os, json, time, hashlib, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from scipy.stats import kendalltau

plt.rcParams["figure.dpi"] = 160
plt.rcParams["font.size"] = 10

ROOT = Path(".")
OUT  = ROOT / "out_final"
OUT.mkdir(exist_ok=True)

# Canonical cohort criteria (edit if you change rubric columns)
CRIT_FULL   = ["Proportion","Niche_Cliff","Drapery","Drapery_Color",
               "Hands","Feet","Head_Hair","Facial","Surface_Erosion","Landscape"]
CRIT_HEAD   = ["Head_Hair","Facial"]           # add "Neck_Lines" if you later score it
CRIT_DRAP   = ["Drapery","Drapery_Color","Hands","Feet"]
COHORT_CRITS = {"Full": CRIT_FULL, "Head": CRIT_HEAD, "Drapery": CRIT_DRAP}

# Optional domain-prior weights (comparison only; not your main ranking)
PRIOR_WEIGHTS = {
    "Proportion":0.18, "Niche_Cliff":0.07, "Drapery":0.16, "Drapery_Color":0.06,
    "Hands":0.09, "Feet":0.06, "Head_Hair":0.13, "Facial":0.09, "Surface_Erosion":0.08, "Landscape":0.08
}

RENAME_MAP = {
    "Niche and Cliff Features":"Niche_Cliff",
    "Niche_Caves":"Niche_Cliff",
    "Head and Hai":"Head_Hair",
    "Surface":"Surface_Erosion",
    "Environment":"Landscape",
}

def barplot(df, x, y, title, fname=None, hline=None):
    fig = plt.figure(figsize=(9.2, 3.6))
    plt.bar(df[x].astype(str), df[y])
    plt.xticks(rotation=60, ha="right", fontsize=8)
    plt.ylabel(y); plt.title(title)
    if hline is not None:
        plt.axhline(hline, ls="--", lw=1, color="gray")
    plt.tight_layout()
    if fname: fig.savefig(OUT/fname, dpi=220)
    plt.show(); plt.close(fig)

def as_str_id(df):
    """Return a **copy** with Image cast to str (avoids SettingWithCopyWarning)."""
    out = df.copy()
    if "Image" in out.columns:
        out.loc[:, "Image"] = out["Image"].astype(str)
    return out


In [ ]:
# Cell 1 — Load, clean, split
rub = pd.read_csv("rubric_scores.csv")
meta = pd.read_csv("images_meta.csv")

rub = rub.rename(columns={k:v for k,v in RENAME_MAP.items() if k in rub.columns})
for c in rub.columns:
    if c != "Image":
        rub[c] = rub[c].replace(["NE","NA","ne","na",""], np.nan)

# ensure numeric where possible
for c in rub.columns:
    if c != "Image":
        rub[c] = pd.to_numeric(rub[c], errors="coerce")

df = rub.merge(meta, on="Image", how="outer", indicator=True)
if (miss := df[df["_merge"]=="left_only"]["Image"].tolist()):
    print("⚠ In rubric but missing meta:", miss)
if (miss := df[df["_merge"]=="right_only"]["Image"].tolist()):
    print("⚠ In meta but missing rubric:", miss)
df = df.drop(columns=["_merge"])

# make sure all expected columns exist
ALL_CRIT = sorted(set(sum(COHORT_CRITS.values(), [])))
for c in ALL_CRIT:
    if c not in df.columns:
        df[c] = np.nan

# split per cohort
cohort_frames = {k: df[df["Cohort"]==k].copy() for k in COHORT_CRITS.keys()}
{ k: len(v) for k,v in cohort_frames.items() }


In [ ]:
# Cell 2 — Scoring functions (robust + NA-aware)

def equal_weight(dfin, crit):
    M = dfin[crit]
    out = dfin[["Image"]].copy()
    out["EqualScore"] = M.mean(axis=1, skipna=True)
    out = as_str_id(out).sort_values("EqualScore", ascending=False).reset_index(drop=True)
    out["Rank_equal"] = np.arange(1, len(out)+1)
    return out

def _entropy_weights(X):
    # X: numeric, no NA, >1 unique per column
    eps = 1e-12
    Xn = (X - X.min()) / (X.max() - X.min()).replace(0,1)
    P = Xn.div(Xn.sum(axis=0)+eps, axis=1) + eps
    E = -(P*np.log(P)).sum(axis=0) / np.log(len(Xn))
    D = 1 - E
    w = D / D.sum()
    return w

def entropy_topsis_guarded(dfin, crit):
    """Guard against small N and zero-variance columns; return (table, weights, kept_crit)."""
    X = dfin[crit].apply(pd.to_numeric, errors="coerce")
    keep = [c for c in crit if X[c].notna().all() and X[c].nunique() > 1]
    if (len(dfin) < 5) or (len(keep) < 3):
        w = pd.Series(dtype=float)
        out = dfin[["Image"]].copy()
        out["TOPSIS_C"] = np.nan
        out["Rank_topsis"] = np.nan
        return as_str_id(out), w, keep

    Xk = X[keep].copy()
    w  = _entropy_weights(Xk)

    # TOPSIS
    eps = 1e-12
    Xn = (Xk - Xk.min()) / (Xk.max() - Xk.min()).replace(0,1)
    V  = Xn * w
    best, worst = V.max(axis=0), V.min(axis=0)
    S_plus  = np.sqrt(((V - best )**2).sum(axis=1))
    S_minus = np.sqrt(((V - worst)**2).sum(axis=1))
    C = S_minus / (S_plus + S_minus + eps)

    out = dfin[["Image"]].copy()
    out["TOPSIS_C"] = C.values
    out = as_str_id(out).sort_values("TOPSIS_C", ascending=False).reset_index(drop=True)
    out["Rank_topsis"] = np.arange(1, len(out)+1)
    return out, w, keep

def borda_NA_aware(dfin, crit):
    """
    Rank only on available criteria per image; convert rank to points in [0,1];
    average points across criteria.
    """
    M = dfin[crit].apply(pd.to_numeric, errors="coerce")
    points = pd.DataFrame(index=dfin.index)
    for c in crit:
        col = M[c]
        ok  = col.dropna().sort_values(ascending=False)
        if ok.empty:
            pts = pd.Series(0.0, index=M.index)
        else:
            ranks = ok.rank(ascending=False, method="average")
            pts_ok = (ranks.max() - ranks) / (ranks.max() - ranks.min() + 1e-12)
            pts = pd.Series(0.0, index=M.index); pts.loc[pts_ok.index] = pts_ok
        points[c] = pts
    out = dfin[["Image"]].copy()
    out["BordaScore"] = points.mean(axis=1, skipna=True)
    out = as_str_id(out).sort_values("BordaScore", ascending=False).reset_index(drop=True)
    out["Rank_borda"] = np.arange(1, len(out)+1)
    return out

def weighted_with_prior(dfin, crit, prior):
    W = {c: prior[c] for c in crit if c in prior}
    Z = sum(W.values()); W = {k:v/Z for k,v in W.items()} if Z>0 else {}
    M = dfin[crit].apply(pd.to_numeric, errors="coerce")

    def wrow(r):
        present = [c for c in crit if not pd.isna(r[c]) and c in W]
        if not present: return np.nan
        z = sum(W[c] for c in present)
        return sum(W[c]*r[c] for c in present)/z

    out = dfin[["Image"]].copy()
    out["PriorScore"] = M.apply(wrow, axis=1)
    out = as_str_id(out).sort_values("PriorScore", ascending=False).reset_index(drop=True)
    out["Rank_prior"] = np.arange(1, len(out)+1)
    return out


In [ ]:
# Cell 3 — Run per cohort + save artifacts
rank_tables = {}
topsis_weights = {}

for cohort, crit in COHORT_CRITS.items():
    dfc = cohort_frames[cohort].copy()
    if dfc.empty:
        print(f"— No images in {cohort} —")
        continue

    eq = equal_weight(dfc, crit)
    barplot(eq, "Image", "EqualScore", f"{cohort}: Equal-weight mean", f"{cohort}_equal.png")

    tp, w, kept = entropy_topsis_guarded(dfc, crit)
    if not w.empty:
        w.to_csv(OUT/f"{cohort}_topsis_entropy_weights.csv")
        print(f"\n{cohort} entropy weights (kept {kept}):\n{w.sort_values(ascending=False)}")
    barplot(tp.fillna(0), "Image", "TOPSIS_C", f"{cohort}: Entropy-TOPSIS closeness (guarded)", f"{cohort}_topsis.png")

    bd = borda_NA_aware(dfc, crit)
    barplot(bd, "Image", "BordaScore", f"{cohort}: Borda (NA-aware)", f"{cohort}_borda.png")

    pr = weighted_with_prior(dfc, crit, PRIOR_WEIGHTS)

    # Merge three (+ prior) — use copies cast to str to avoid dtype warnings
    tab = (as_str_id(eq)[["Image","EqualScore","Rank_equal"]]
           .merge(as_str_id(tp)[["Image","TOPSIS_C","Rank_topsis"]], on="Image", how="outer")
           .merge(as_str_id(bd)[["Image","BordaScore","Rank_borda"]], on="Image", how="outer")
           .merge(as_str_id(pr)[["Image","PriorScore","Rank_prior"]], on="Image", how="outer"))

    tab.to_csv(OUT/f"{cohort}_three_methods.csv", index=False)
    rank_tables[cohort] = tab

    # Print Top-3 safely
    def topn(col):
        return ", ".join(map(str, tab.sort_values(col, na_position="last")["Image"].head(3)))
    print(f"\n{cohort} Top-3 (Equal):  {topn('Rank_equal')}")
    print(f"{cohort} Top-3 (TOPSIS): {topn('Rank_topsis')}")
    print(f"{cohort} Top-3 (Borda):  {topn('Rank_borda')}")
    print(f"{cohort} Top-3 (Prior):  {topn('Rank_prior')}")


In [ ]:
# Cell 4 — Inter-rater reliability (κ), optional
R2 = Path("rubric_scores_rater2.csv")
if not R2.exists():
    print("ℹ No rubric_scores_rater2.csv — skipping reliability.")
else:
    rub2 = pd.read_csv(R2).rename(columns={k:v for k,v in RENAME_MAP.items() if k in rub2.columns})
    for c in rub2.columns:
        if c != "Image":
            rub2[c] = rub2[c].replace(["NE","NA","ne","na",""], np.nan)
            rub2[c] = pd.to_numeric(rub2[c], errors="coerce")

    merged = df[["Image"]+ALL_CRIT].merge(rub2[["Image"]+ALL_CRIT], on="Image", suffixes=("_r1","_r2"))
    rows = []
    for crit in ALL_CRIT:
        a = merged[f"{crit}_r1"]; b = merged[f"{crit}_r2"]
        mask = (~a.isna()) & (~b.isna())
        kap = cohen_kappa_score(a[mask], b[mask]) if mask.sum() else np.nan
        rows.append({"Criterion":crit,"Kappa":kap,"N_pairs":int(mask.sum())})
    kdf = pd.DataFrame(rows).sort_values("Criterion")
    kdf.to_csv(OUT/"reliability_kappa.csv", index=False)
    print(kdf)

    fig = plt.figure(figsize=(9.2,3.6))
    plt.bar(kdf["Criterion"], kdf["Kappa"].fillna(0))
    plt.xticks(rotation=60, ha="right", fontsize=8)
    plt.axhline(0.60, ls="--", lw=1, color="gray")
    plt.title("Inter-rater reliability (Cohen’s κ)")
    plt.tight_layout(); fig.savefig(OUT/"fig_kappa_per_criterion.png", dpi=220)
    plt.show(); plt.close(fig)


In [ ]:
# Cell 5 — Kendall τ across methods (per cohort)
for cohort, tab in rank_tables.items():
    use = tab.dropna(subset=["Rank_equal","Rank_borda"], how="all").copy()
    taus = {}
    cols = ["Rank_equal","Rank_topsis","Rank_borda","Rank_prior"]
    # pairwise τ for columns that exist & not all-NaN
    pairs = [("equal","topsis"),("equal","borda"),("equal","prior"),
             ("topsis","borda"),("topsis","prior"),("borda","prior")]
    for a,b in pairs:
        ca, cb = f"Rank_{a}", f"Rank_{b}"
        if ca in use.columns and cb in use.columns:
            s = use[[ca,cb]].dropna()
            taus[f"{a}_vs_{b}"] = kendalltau(s[ca], s[cb]).correlation if len(s)>1 else np.nan
    print(f"\n{cohort} — Kendall τ:\n", pd.Series(taus))
    pd.Series(taus).to_csv(OUT/f"{cohort}_kendall_tau.csv")


In [ ]:
# Cell 6 — Stability

def prior_with_perturb(prior, crit, delta):
    W = {c: prior[c] for c in crit if c in prior}
    # bump top-2 weights
    top2 = sorted(W, key=W.get, reverse=True)[:2]
    Wp = W.copy()
    for c in top2:
        Wp[c] = max(Wp[c]*(1+delta), 0)
    Z = sum(Wp.values()); Wp = {k:v/Z for k,v in Wp.items()} if Z>0 else W
    return Wp, top2

def jitter_series(s, sigma=0.10):
    x = s.astype(float).values
    x = np.clip(x + np.random.normal(0, sigma, size=x.shape), 0.0, 1.0)
    return pd.Series(x, index=s.index)

B = 1000
np.random.seed(7)

for cohort, crit in COHORT_CRITS.items():
    dfc = cohort_frames[cohort]
    if dfc.empty: continue

    # (A) weight perturbation on domain prior
    base = weighted_with_prior(dfc, crit, PRIOR_WEIGHTS)
    base_top3 = set(base["Image"].head(3))
    rows = []
    for delta in (0.05, 0.10):
        Wp, top2 = prior_with_perturb(PRIOR_WEIGHTS, crit, delta)
        sw = weighted_with_prior(dfc, crit, Wp)
        overlap = len(base_top3 & set(sw["Image"].head(3))) / 3.0
        rows.append({"Delta": delta, "Top2Touched": ",".join(top2), "Top3_overlap": overlap})
    stab = pd.DataFrame(rows)
    stab.to_csv(OUT/f"{cohort}_stability_weight_perturb.csv", index=False)
    barplot(stab, "Delta", "Top3_overlap", f"{cohort}: Top-3 overlap under prior-weight perturbation",
            f"{cohort}_stability_weight_perturb.png", hline=0.66)

    # (B) bootstrap jitter on rubric (TOPSIS+Borda)
    names = list(dfc["Image"].astype(str))
    M0 = dfc[crit].apply(pd.to_numeric, errors="coerce")
    top3_t, top3_b = dict.fromkeys(names,0), dict.fromkeys(names,0)

    for b in range(B):
        M = M0.copy()
        for c in crit:
            col = M[c]
            mask = ~col.isna()
            if mask.any():
                M.loc[mask, c] = jitter_series(col[mask], sigma=0.10).values
        tmp = dfc.copy()
        tmp.loc[:, crit] = M[crit]

        tp, _, _ = entropy_topsis_guarded(tmp, crit)
        bd = borda_NA_aware(tmp, crit)
        for img in set(tp["Image"].head(3).astype(str)): top3_t[img] += 1
        for img in set(bd["Image"].head(3).astype(str)): top3_b[img] += 1

    ft = pd.DataFrame({"Image":names,
                       "Top3_freq_TOPSIS":[top3_t[n]/B for n in names],
                       "Top3_freq_Borda":[top3_b[n]/B for n in names]})
    ft = ft.sort_values(["Top3_freq_TOPSIS","Top3_freq_Borda"], ascending=False)
    ft.to_csv(OUT/f"{cohort}_bootstrap_top3_freq.csv", index=False)

    # lollipop (single-plot)
    fig = plt.figure(figsize=(9.2,3.6))
    freq_mean = ft[["Top3_freq_TOPSIS","Top3_freq_Borda"]].mean(axis=1)
    markerline, stemlines, baseline = plt.stem(range(len(ft)), freq_mean, use_line_collection=True)
    plt.xticks(range(len(ft)), ft["Image"].astype(str), rotation=60, ha="right", fontsize=8)
    plt.ylabel("Top-3 frequency (mean of methods)")
    plt.title(f"{cohort}: stability under score jitter (±0.10)")
    plt.tight_layout(); fig.savefig(OUT/f"{cohort}_bootstrap_lollipop.png", dpi=220)
    plt.show(); plt.close(fig)


In [ ]:
# Cell 7 — Manifest + concise markdown report

def file_md5(p):
    try:
        with open(p, "rb") as f:
            h = hashlib.md5()
            for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

manifest = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "files": {
        "rubric_scores.csv": file_md5("rubric_scores.csv"),
        "images_meta.csv":   file_md5("images_meta.csv"),
        "constraints.csv":   file_md5("constraints.csv") if Path("constraints.csv").exists() else None,
        "rubric_scores_rater2.csv": file_md5("rubric_scores_rater2.csv") if Path("rubric_scores_rater2.csv").exists() else None
    },
    "cohorts": {k: len(v) for k,v in cohort_frames.items()},
    "methods": ["Equal mean (NA-renorm)","Entropy-TOPSIS (guarded)","Borda (NA-aware)","Domain-prior (comparison)"],
    "bootstrap_B": 1000
}
with open(OUT/"manifest.json","w") as f: json.dump(manifest, f, indent=2)
print("Manifest → out_final/manifest.json")

# Build a short report
lines = ["# Evaluation Summary\n"]
for cohort, tab in rank_tables.items():
    def t3(col):
        return ", ".join(map(str, tab.sort_values(col, na_position="last").Image.head(3)))
    lines += [
        f"## {cohort}\n",
        f"**Top-3 (Equal):** {t3('Rank_equal')}\n",
        f"**Top-3 (TOPSIS):** {t3('Rank_topsis')}\n",
        f"**Top-3 (Borda):** {t3('Rank_borda')}\n",
        f"**Top-3 (Prior):** {t3('Rank_prior')}\n",
        f"Figures: `{cohort}_equal.png`, `{cohort}_topsis.png`, `{cohort}_borda.png`.\n"
    ]
    stab = OUT/f"{cohort}_stability_weight_perturb.csv"
    if stab.exists():
        s = pd.read_csv(stab)
        sstr = "; ".join([f"Δ={r.Delta:+.0%} → overlap={r.Top3_overlap:.2f}" for _,r in s.iterrows()])
        lines.append(f"**Weight-perturbation overlap:** {sstr}\n")
        lines.append(f"Figure: `{cohort}_stability_weight_perturb.png`.\n")
    boot = OUT/f"{cohort}_bootstrap_top3_freq.csv"
    if boot.exists():
        lines.append(f"Bootstrap top-3 frequencies: see `{cohort}_bootstrap_top3_freq.csv` and `{cohort}_bootstrap_lollipop.png`.\n")

kappa_csv = OUT/"reliability_kappa.csv"
if kappa_csv.exists():
    kdf = pd.read_csv(kappa_csv)
    lines += ["\n## Reliability (Cohen’s κ)\n",
              f"Median κ across criteria: {kdf['Kappa'].dropna().median():.2f}. Threshold line at 0.60 shown in `fig_kappa_per_criterion.png`.\n"]

with open(OUT/"report.md","w",encoding="utf-8") as f: f.write("\n".join(lines))
print("Report → out_final/report.md")


In [ ]:
# Cell V — Restore rich visuals using current in-memory outputs (rank_tables, cohort_frames, OUT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

def crit_list_for(cohort):
    return {
        "Full": ["Proportion","Niche_Cliff","Drapery","Drapery_Color","Hands","Feet","Head_Hair","Facial","Surface_Erosion","Landscape"],
        "Head": ["Head_Hair","Facial"],
        "Drapery": ["Drapery","Drapery_Color","Hands","Feet"],
    }[cohort]

# ---------- Heatmap (images × criteria) ----------
def plot_heatmap(cohort, dfin, crit):
    M = dfin[crit].apply(pd.to_numeric, errors="coerce")
    M = M.fillna(M.median())  # simple impute for display
    fig = plt.figure(figsize=(8.8, 4.6))
    plt.imshow(M.values, aspect="auto")
    plt.colorbar()
    plt.yticks(range(len(dfin)), dfin["Image"].astype(str), fontsize=7)
    plt.xticks(range(len(crit)), crit, rotation=60, ha="right", fontsize=8)
    plt.title(f"{cohort}: rubric heatmap (images × criteria)")
    plt.tight_layout()
    fig.savefig(OUT/f"{cohort}_viz_heatmap.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Bump / rank trajectories across methods ----------
def plot_bump(cohort, tab):
    methods = [c for c in ["Rank_equal","Rank_topsis","Rank_borda","Rank_prior"] if c in tab.columns]
    if len(methods) < 2: 
        print(f"ℹ {cohort}: not enough ranking methods to plot bump."); return
    small = tab.assign(_best=tab[methods].min(axis=1)).sort_values("_best").head(6)
    x = np.arange(len(methods))
    fig = plt.figure(figsize=(9.2, 4.0))
    for _,r in small.iterrows():
        y = [r[m] for m in methods]
        plt.plot(x, y, marker="o", linewidth=1.5)
        plt.text(x[0]-0.05, y[0], str(r["Image"]), fontsize=7, ha="right", va="center")
    plt.gca().invert_yaxis()
    plt.xticks(x, [m.replace("Rank_","").upper() for m in methods])
    plt.ylabel("Rank (1 = best)")
    plt.title(f"{cohort}: rank trajectories (bump chart)")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort}_viz_bump_ranks.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Radar for top Equal-mean image ----------
def plot_radar_top1(cohort, dfin, crit):
    M = dfin[crit].apply(pd.to_numeric, errors="coerce")
    prof = M.mean(axis=1, skipna=True)
    idx = prof.sort_values(ascending=False).index[0]
    top_img = dfin.loc[idx, "Image"]
    row = M.loc[idx].fillna(M.median())

    # normalize per-criterion to [0,1]
    col_min, col_max = M.min(), M.max().replace(0,1)
    vals = ((row - col_min) / (col_max - col_min)).values.astype(float)

    import numpy as np
    N = len(crit)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    vals = np.concatenate((vals, [vals[0]]))
    angles += angles[:1]

    fig = plt.figure(figsize=(5.8,5.8))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, vals, linewidth=2)
    ax.fill(angles, vals, alpha=0.25)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(crit, fontsize=8)
    ax.set_yticklabels([])
    ax.set_title(f"{cohort}: radar of top Equal image ({top_img})")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort}_viz_radar_top_equal.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Violin: criterion discriminativeness ----------
def plot_violin(cohort, dfin, crit):
    M = dfin[crit].apply(pd.to_numeric, errors="coerce")
    data = [M[c].dropna().values for c in crit]
    if all(len(d)==0 for d in data):
        print(f"ℹ {cohort}: no data for violin."); return
    fig = plt.figure(figsize=(9.6, 3.8))
    plt.violinplot(data, showmeans=True, showextrema=False)
    plt.xticks(range(1, len(crit)+1), crit, rotation=60, ha="right", fontsize=8)
    plt.ylabel("Score distribution")
    plt.title(f"{cohort}: discriminativeness per criterion (violin)")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort}_viz_violin_crit.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Consensus grid: Top-3 membership across methods ----------
def plot_consensus_grid(cohort, tab):
    methods = [c for c in ["Rank_equal","Rank_topsis","Rank_borda","Rank_prior"] if c in tab.columns]
    if not methods: 
        print(f"ℹ {cohort}: no methods to draw grid."); return
    mat = []
    for _,r in tab.iterrows():
        mat.append([1 if (not pd.isna(r[m]) and r[m] <= 3) else 0 for m in methods])
    mat = np.array(mat)
    fig = plt.figure(figsize=(6.6, max(3.2, 0.25*len(tab))))
    for i in range(len(tab)):
        for j in range(len(methods)):
            if mat[i,j] == 1: plt.scatter(j, i, s=80)
    plt.yticks(range(len(tab)), tab["Image"].astype(str), fontsize=7)
    plt.xticks(range(len(methods)), [m.replace("Rank_","").upper() for m in methods])
    plt.gca().invert_yaxis()
    plt.title(f"{cohort}: consensus grid (Top-3 per method)")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort}_viz_consensus_grid.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Slopegraph: Equal vs TOPSIS ----------
def plot_slope_equal_topsis(cohort, tab):
    if not {"Rank_equal","Rank_topsis"}.issubset(tab.columns): 
        print(f"ℹ {cohort}: need Equal and TOPSIS ranks for slopegraph."); return
    small = tab.assign(_best=tab[["Rank_equal","Rank_topsis"]].min(axis=1)).sort_values("_best").head(8)
    fig = plt.figure(figsize=(6.8, 4.0))
    for _,r in small.iterrows():
        plt.plot([0,1], [r["Rank_equal"], r["Rank_topsis"]], marker="o")
        plt.text(-0.02, r["Rank_equal"], str(r["Image"]), fontsize=7, ha="right", va="center")
    plt.gca().invert_yaxis()
    plt.xticks([0,1], ["EQUAL","TOPSIS"])
    plt.ylabel("Rank (1 = best)")
    plt.title(f"{cohort}: slopegraph (Equal vs TOPSIS)")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort}_viz_slope_equal_topsis.png", dpi=220)
    plt.show(); plt.close(fig)

# ---------- Drive per cohort ----------
for cohort, tab in rank_tables.items():
    dfin = cohort_frames[cohort].copy()
    if dfin.empty: 
        print(f"— No images in {cohort} —"); 
        continue
    crit = crit_list_for(cohort)

    plot_heatmap(cohort, dfin, crit)
    plot_bump(cohort, tab.copy())
    plot_radar_top1(cohort, dfin, crit)
    plot_violin(cohort, dfin, crit)
    plot_consensus_grid(cohort, tab.copy())
    plot_slope_equal_topsis(cohort, tab.copy())


In [ ]:
# V1 — UpSet-style Top-3 overlap across Equal / TOPSIS / Borda / Prior
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from itertools import product

def upset_top3(tab, cohort_name):
    methods = [m for m in ["Rank_equal","Rank_topsis","Rank_borda","Rank_prior"] if m in tab.columns]
    if len(methods) < 2: 
        print(f"{cohort_name}: need ≥2 methods for UpSet"); 
        return
    # membership dict per method
    mem = {m: set(tab.sort_values(m, na_position="last").Image.head(3).astype(str)) for m in methods}
    # build all combinations
    rows=[]
    for mask in product([0,1], repeat=len(methods)):
        if sum(mask)==0: continue
        active=[methods[i] for i,b in enumerate(mask) if b]
        inter = set.intersection(*(mem[m] for m in active)) if active else set()
        rows.append({"mask":mask,"k":sum(mask),"size":len(inter)})
    df = pd.DataFrame(rows)
    if df["size"].sum()==0:
        print(f"{cohort_name}: no intersections"); return
    # plot
    fig = plt.figure(figsize=(9.4, 4.8))
    ax_bar = plt.axes([0.08,0.40,0.88,0.55])
    ax_mat = plt.axes([0.08,0.10,0.88,0.22])

    order = df.sort_values(["k","size"], ascending=[True,False]).index
    sizes = df.loc[order,"size"].values
    ax_bar.bar(range(len(order)), sizes)
    ax_bar.set_ylabel("Intersection size")
    ax_bar.set_title(f"{cohort_name}: Top-3 overlaps (Equal/TOPSIS/Borda/Prior)")
    ax_bar.set_xticks([])

    mat = np.array(list(df.loc[order,"mask"].values))
    for i in range(mat.shape[1]):
        ax_mat.scatter(range(mat.shape[0]), [i]*mat.shape[0], s=110*(mat[:,i]), marker="o")
    ax_mat.set_yticks(range(len(methods))); 
    ax_mat.set_yticklabels([m.replace("Rank_","").upper() for m in methods])
    ax_mat.set_xticks(range(mat.shape[0])); ax_mat.set_xticklabels([])
    ax_mat.set_xlim(-0.5, mat.shape[0]-0.5)
    fig.savefig(OUT/f"{cohort_name}_viz_upset_top3.png", dpi=220, bbox_inches="tight")
    plt.show(); plt.close(fig)

for cohort, tab in rank_tables.items():
    upset_top3(tab.copy(), cohort)


In [ ]:
# V2 — Alluvial (thick bump) for Top-8: Equal → TOPSIS → Borda
import numpy as np, matplotlib.pyplot as plt

def alluvial_three(tab, cohort_name):
    cols = [c for c in ["Rank_equal","Rank_topsis","Rank_borda"] if c in tab.columns]
    if len(cols)<2: 
        print(f"{cohort_name}: need ≥2 rank methods for alluvial"); 
        return
    t = tab.dropna(subset=[cols[0]]).copy()
    t["_best"] = t[cols].min(axis=1, skipna=True)
    t = t.sort_values("_best").head(8)
    x = np.arange(len(cols))
    fig = plt.figure(figsize=(8.8, 4.2))
    for _,r in t.iterrows():
        y=[r[c] if not np.isnan(r[c]) else np.nan for c in cols]
        w = 3.0 - 0.25*(r["_best"]-1)  # thicker line for better overall rank
        plt.plot(x, y, linewidth=w, alpha=0.85)
        plt.text(x[0]-0.05, y[0], str(r["Image"]), fontsize=8, ha="right", va="center")
    plt.gca().invert_yaxis()
    plt.xticks(x, [c.replace("Rank_","").upper() for c in cols])
    plt.ylabel("Rank (1 = best)"); 
    plt.title(f"{cohort_name}: alluvial flow of ranks")
    plt.tight_layout()
    plt.savefig(OUT/f"{cohort_name}_viz_alluvial.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, tab in rank_tables.items():
    alluvial_three(tab.copy(), cohort)


In [ ]:
# V3 — Dumbbell plot: Equal vs TOPSIS ranks (Top-12 by best rank)
import numpy as np, matplotlib.pyplot as plt, pandas as pd

def dumbbell_equal_topsis(tab, cohort_name):
    if not {"Rank_equal","Rank_topsis"}.issubset(tab.columns):
        print(f"{cohort_name}: need Equal and TOPSIS ranks"); return
    t = tab.copy()
    t["_best"] = t[["Rank_equal","Rank_topsis"]].min(axis=1)
    t = t.sort_values("_best").head(12)
    t = t.sort_values("Rank_equal")  # order left axis
    fig = plt.figure(figsize=(7.5, 5.6))
    y = np.arange(len(t))
    plt.hlines(y, t["Rank_equal"], t["Rank_topsis"], lw=2)
    plt.plot(t["Rank_equal"], y, 'o', label="EQUAL")
    plt.plot(t["Rank_topsis"], y, 'o', label="TOPSIS")
    big = (t["Rank_equal"]-t["Rank_topsis"]).abs()>=3
    plt.hlines(y[big], t["Rank_equal"][big], t["Rank_topsis"][big], lw=4, alpha=0.5)
    plt.gca().invert_xaxis(); plt.yticks(y, t["Image"].astype(str))
    plt.xlabel("Rank (1 = best)"); plt.legend(loc="lower right")
    plt.title(f"{cohort_name}: Equal vs TOPSIS (dumbbell)")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_dumbbell_equal_topsis.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, tab in rank_tables.items():
    dumbbell_equal_topsis(tab.copy(), cohort)


In [ ]:
# V4 — Waterfall: contributions from ideal=1.0 to final Equal mean (Top-1 image)
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def waterfall_penalties(dfc, crit, cohort_name):
    eq = dfc[crit].mean(axis=1, skipna=True)
    idx = eq.sort_values(ascending=False).index[0]
    top_img = dfc.loc[idx, "Image"]
    row = dfc.loc[idx, crit].astype(float)
    penalties = (1.0 - row).fillna(0.0)  # missing criteria contribute 0 to mean; you can change
    order = penalties.sort_values(ascending=False).index
    vals = penalties[order].values

    fig = plt.figure(figsize=(9.0, 4.0))
    cum = 0
    x = range(len(order))
    for i,v in zip(x, vals):
        c = "tab:red" if v>0 else "tab:green"
        plt.bar(i, v, bottom=cum, color=c, alpha=0.8)
        cum += v
    plt.xticks(x, order, rotation=60, ha="right")
    plt.ylabel("Penalty (1 − score)"); 
    target = 1.0 - penalties.mean()  # equals the equal-mean if all criteria present
    plt.title(f"{cohort_name}: penalty waterfall for {top_img} (Equal≈{target:.2f})")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_waterfall_penalties.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, dfc in cohort_frames.items():
    crit = COHORT_CRITS[cohort]
    if not dfc.empty: waterfall_penalties(dfc.copy(), crit, cohort)


In [ ]:
# V5 — Compliance wheel (radial stacked bars) for the top Equal image
import numpy as np, matplotlib.pyplot as plt, pandas as pd

def compliance_wheel(dfc, crit, cohort_name):
    M = dfc[crit].apply(pd.to_numeric, errors="coerce")
    eq = M.mean(axis=1, skipna=True)
    ix = eq.sort_values(ascending=False).index[0]
    top_img = dfc.loc[ix,"Image"]
    row = M.loc[ix].fillna(0)

    theta = np.linspace(0, 2*np.pi, len(crit), endpoint=False)
    width = 2*np.pi/len(crit)
    fig = plt.figure(figsize=(6.2,6.2))
    ax = plt.subplot(111, polar=True)
    # three stacked rings for 0 / 0.5 / 1
    base = np.zeros_like(theta)
    half = np.where(row>=0.5, 0.5, np.clip(row,0,0.5))
    full = np.where(row==1.0, 0.5, 0.0)  # extra to reach 1
    ax.bar(theta, half, width=width*0.85, bottom=base, alpha=0.55)
    ax.bar(theta, full, width=width*0.85, bottom=half, alpha=0.85)
    ax.set_xticks(theta); ax.set_xticklabels(crit, fontsize=8)
    ax.set_yticklabels([]); 
    ax.set_title(f"{cohort_name}: compliance wheel — {top_img}")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_compliance_wheel.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, dfc in cohort_frames.items():
    crit = COHORT_CRITS[cohort]
    if not dfc.empty: compliance_wheel(dfc.copy(), crit, cohort)


In [ ]:
# V6 — Badge strip: tiny glyph row per image (✓ / ~ / ✗ = 1 / 0.5 / 0)
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def badge_strip(dfc, crit, cohort_name):
    M = dfc[["Image"]+crit].copy()
    # map values to -1, 0, +1 colors
    def badge(v):
        if pd.isna(v): return 0.2
        if v>=0.999: return 1.0
        if v>=0.499: return 0.6
        return 0.0
    A = np.array([[badge(M.loc[i,c]) for c in crit] for i in M.index])
    fig = plt.figure(figsize=(9.6, max(3.0, 0.35*len(M))))
    plt.imshow(A, aspect="auto", vmin=0, vmax=1)
    plt.colorbar(ticks=[0,0.6,1], label="0 / 0.5 / 1")
    plt.yticks(range(len(M)), M["Image"].astype(str), fontsize=7)
    plt.xticks(range(len(crit)), crit, rotation=60, ha="right", fontsize=8)
    plt.title(f"{cohort_name}: rubric micro-glyphs per image")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_badge_strip.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, dfc in cohort_frames.items():
    crit = COHORT_CRITS[cohort]
    if not dfc.empty: badge_strip(dfc.copy(), crit, cohort)


In [ ]:
# V7 — Ridgeline-style stacked histograms per criterion
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def ridgeline(dfc, crit, cohort_name):
    M = dfc[crit].apply(pd.to_numeric, errors="coerce")
    bins = np.linspace(0,1,6)
    fig = plt.figure(figsize=(8.8, 0.55*len(crit)+1.5))
    for i,c in enumerate(crit):
        vals = M[c].dropna().values
        if len(vals)==0: continue
        hist,edges = np.histogram(vals, bins=bins, density=True)
        x = (edges[:-1]+edges[1:])/2
        y = hist / (hist.max() if hist.max()>0 else 1) + i
        plt.fill_between(x, i, y, alpha=0.6, step="mid")
        plt.text(1.02, i+0.05, c, fontsize=8, ha="left", va="bottom")
    plt.yticks([]); plt.xlabel("Score"); 
    plt.title(f"{cohort_name}: criterion score distributions (stacked)")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_ridgeline.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, dfc in cohort_frames.items():
    crit = COHORT_CRITS[cohort]
    if not dfc.empty: ridgeline(dfc.copy(), crit, cohort)


In [ ]:
# V8 — Stability cartogram: Top-3 frequency with 95% binomial CI whiskers
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import sqrt

def cartogram_top3(cohort):
    p = OUT/f"{cohort}_bootstrap_top3_freq.csv"
    if not p.exists():
        print(f"{cohort}: no bootstrap file — run your bootstrap cell first."); return
    ft = pd.read_csv(p)
    # mean of methods
    ft["MeanFreq"] = ft[["Top3_freq_TOPSIS","Top3_freq_Borda"]].mean(axis=1)
    # binomial 95% CI (Wilson approx) — B known from your bootstrap cell header (default 1000)
    B = 1000
    p_ = ft["MeanFreq"].values
    z = 1.96
    ci = z*np.sqrt((p_*(1-p_))/B)
    fig = plt.figure(figsize=(8.8, 4.6))
    order = np.argsort(-p_)
    x = np.arange(len(ft))
    plt.errorbar(x, p_[order], yerr=ci[order], fmt='o')
    plt.xticks(x, ft["Image"].astype(str).values[order], rotation=60, ha="right")
    plt.ylim(-0.02, 1.02)
    plt.ylabel("Top-3 frequency (mean of methods)")
    plt.title(f"{cohort}: Top-3 stability with 95% CI")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort}_viz_cartogram_stability.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort in rank_tables.keys():
    cartogram_top3(cohort)


In [ ]:
# V9 — Decision map: Equal vs TOPSIS
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def decision_map(tab, cohort_name):
    if not {"EqualScore","TOPSIS_C"}.issubset(tab.columns): 
        print(f"{cohort_name}: need Equal and TOPSIS metrics"); return
    t = tab.copy()
    fig = plt.figure(figsize=(6.6, 5.2))
    plt.scatter(t["EqualScore"], t["TOPSIS_C"])
    for _,r in t.iterrows():
        plt.text(r["EqualScore"]+0.002, r["TOPSIS_C"], str(r["Image"]), fontsize=7)
    plt.axvline(0.80, ls="--", lw=1, color="gray")
    plt.axhline(t["TOPSIS_C"].quantile(0.75), ls="--", lw=1, color="gray")
    plt.xlabel("Equal-weight mean"); plt.ylabel("TOPSIS closeness")
    plt.title(f"{cohort_name}: decision map (agreement quadrant top-right)")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_decision_map.png", dpi=220)
    plt.show(); plt.close(fig)

for cohort, tab in rank_tables.items():
    decision_map(tab.copy(), cohort)


In [ ]:
# V10 — Entropy-weight donut; variance bars if *_weights_boot.csv exists
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def donut_weights(cohort_name):
    p = OUT/f"{cohort_name}_topsis_entropy_weights.csv"
    if not p.exists():
        print(f"{cohort_name}: no entropy weights file"); return
    w = pd.read_csv(p, header=None, names=["Criterion","Weight"]) if p.suffix==".csv" else None
    if w is None:
        print(f"{cohort_name}: bad weights format"); return
    w = w.sort_values("Weight", ascending=False)
    vals = w["Weight"].values; labels = w["Criterion"].tolist()

    fig = plt.figure(figsize=(6.2,6.2))
    wedges, _ = plt.pie(vals, startangle=90)
    plt.gca().add_artist(plt.Circle((0,0), 0.55, color="white"))
    plt.legend(wedges, [f"{l}  ({v:.2f})" for l,v in zip(labels, vals)], loc="center", bbox_to_anchor=(0.5,0.5))
    plt.title(f"{cohort_name}: entropy weights (donut)")
    plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_weights_donut.png", dpi=220)
    plt.show(); plt.close(fig)

    # optional: variance bars if you later generate bootstrap weights into {cohort}_weights_boot.csv
    pb = OUT/f"{cohort_name}_weights_boot.csv"
    if pb.exists():
        wb = pd.read_csv(pb)  # columns: Criterion, Mean, SD (expected)
        fig = plt.figure(figsize=(7.8,4.0))
        x = np.arange(len(wb))
        plt.bar(x, wb["Mean"], yerr=wb["SD"], alpha=0.8)
        plt.xticks(x, wb["Criterion"], rotation=60, ha="right")
        plt.ylabel("Weight ± SD"); plt.title(f"{cohort_name}: entropy weight stability")
        plt.tight_layout(); plt.savefig(OUT/f"{cohort_name}_viz_weights_variance.png", dpi=220)
        plt.show(); plt.close(fig)

for cohort in rank_tables.keys():
    donut_weights(cohort)
